# Visual Question Answering (VQA) Training on Kaggle with VQA v2

In [1]:

!pip install -q transformers datasets accelerate tqdm pillow torch matplotlib


## 1. Imports and GPU Setup
We'll import all necessary packages and verify that CUDA GPU acceleration is active on Kaggle.


In [2]:

import os
import io
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, IterableDataset
from PIL import Image
from tqdm.auto import tqdm
# UPGRADE: Using CLIP Vision Model and CLIP Image Processor instead of ViT
from transformers import CLIPImageProcessor, GPT2Tokenizer, CLIPVisionModel, GPT2LMHeadModel
from datasets import load_dataset

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU acceleration available: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('WARNING: GPU not available. Training will be extremely slow. Please enable GPU in settings.')


GPU acceleration available: Tesla T4


## 2. Dataset Definition for VQA v2
We define a custom PyTorch `Dataset` wrapper (`VQAv2Dataset`) to load images, questions, and multiple-choice answers from the Hugging Face `datasets` format.

For each sample:
1. We read the image and convert it to **RGB**.
2. We extract the visual features using `ViTImageProcessor`.
3. We construct the sequence text: `Question: {question} Answer: {answer}`.
4. We mask out the question prompt tokens by assigning `-100` in the labels. This ensures the model is only penalized for errors in the predicted answer tokens (and not the question prompt).


In [3]:

class VQAv2Dataset(Dataset):
    def __init__(self, hf_dataset, vit_image_processor, tokenizer, max_length=40):
        self.dataset = hf_dataset
        self.image_processor = vit_image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        img = item['image']

        # Convert image to RGB (some VQA images can be grayscale)
        if img.mode != 'RGB':
            img = img.convert('RGB')

        question = item['question']

        # SAFETY FIX: Handle case where answer might be None (like in the test split)
        answer = item.get('multiple_choice_answer', '')
        if answer is None:
            answer = ''

        # 1. Preprocess image using CLIP Vision Image Processor
        image_inputs = self.image_processor(images=img, return_tensors='pt')
        pixel_values = image_inputs.pixel_values.squeeze(0) # shape: (3, 224, 224)

        # 2. Build question + answer sequence tokens separately to prevent tokenizer alignment bugs
        prompt_text = f'Question: {question} Answer: '
        answer_text = f'{answer}{self.tokenizer.eos_token}'

        prompt_enc = self.tokenizer(prompt_text, add_special_tokens=False)
        answer_enc = self.tokenizer(answer_text, add_special_tokens=False)

        prompt_ids = prompt_enc['input_ids']
        answer_ids = answer_enc['input_ids']

        input_ids = prompt_ids + answer_ids

        # Generate labels: mask out prompt tokens with -100 exactly
        labels = [-100] * len(prompt_ids) + answer_ids

        # Pad/truncate to max_length
        if len(input_ids) > self.max_length:
            input_ids = input_ids[:self.max_length]
            labels = labels[:self.max_length]
            attention_mask = [1] * self.max_length
        else:
            pad_len = self.max_length - len(input_ids)
            attention_mask = [1] * len(input_ids) + [0] * pad_len
            input_ids = input_ids + [self.tokenizer.pad_token_id] * pad_len
            labels = labels + [-100] * pad_len

        return {
            'pixel_values': pixel_values,
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
            'question': question,
            'answer': answer
        }

class VQAStreamingDataset(IterableDataset):
    def __init__(self, hf_iterable_dataset, vit_image_processor, tokenizer, max_length=40):
        self.dataset = hf_iterable_dataset
        self.image_processor = vit_image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __iter__(self):
        for item in self.dataset:
            img = item['image']

            # Convert image to RGB (some VQA images can be grayscale)
            if img.mode != 'RGB':
                img = img.convert('RGB')

            question = item['question']

            # SAFETY FIX: Handle case where answer might be None (like in the test split)
            answer = item.get('multiple_choice_answer', '')
            if answer is None:
                answer = ''

            # 1. Preprocess image using CLIP Vision Image Processor
            image_inputs = self.image_processor(images=img, return_tensors='pt')
            pixel_values = image_inputs.pixel_values.squeeze(0) # shape: (3, 224, 224)

            # 2. Build question + answer sequence tokens separately to prevent tokenizer alignment bugs
            prompt_text = f'Question: {question} Answer: '
            answer_text = f'{answer}{self.tokenizer.eos_token}'

            prompt_enc = self.tokenizer(prompt_text, add_special_tokens=False)
            answer_enc = self.tokenizer(answer_text, add_special_tokens=False)

            prompt_ids = prompt_enc['input_ids']
            answer_ids = answer_enc['input_ids']

            input_ids = prompt_ids + answer_ids

            # Generate labels: mask out prompt tokens with -100 exactly
            labels = [-100] * len(prompt_ids) + answer_ids

            # Pad/truncate to max_length
            if len(input_ids) > self.max_length:
                input_ids = input_ids[:self.max_length]
                labels = labels[:self.max_length]
                attention_mask = [1] * self.max_length
            else:
                pad_len = self.max_length - len(input_ids)
                attention_mask = [1] * len(input_ids) + [0] * pad_len
                input_ids = input_ids + [self.tokenizer.pad_token_id] * pad_len
                labels = labels + [-100] * pad_len

            yield {
                'pixel_values': pixel_values,
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
                'question': question,
                'answer': answer
            }


## 3. VQA Model Definition
We define the `VQAProjector` (MLP head mapping ViT dim to GPT-2 dim) and the `ViTLLMFusionModel` which combines the visual patch embeddings from ViT with the text embeddings before running through GPT-2 transformer blocks.


In [4]:

class VQAProjector(nn.Module):
    def __init__(self, vit_dim=768, llm_dim=768, use_mlp=True):
        super().__init__()
        if use_mlp:
            self.proj = nn.Sequential(
                nn.Linear(vit_dim, llm_dim),
                nn.GELU(),
                nn.Linear(llm_dim, llm_dim)
            )
        else:
            self.proj = nn.Linear(vit_dim, llm_dim)

    def forward(self, x):
        return self.proj(x)

class ViTLLMFusionModel(nn.Module):
    # UPGRADE: Support CLIPVisionModel by default
    def __init__(self, vit_model_name='openai/clip-vit-base-patch32', llm_model_name='gpt2', use_mlp=True):
        super().__init__()
        self.vit = CLIPVisionModel.from_pretrained(vit_model_name)
        self.llm = GPT2LMHeadModel.from_pretrained(llm_model_name)

        # Freeze vision backbone parameters
        for param in self.vit.parameters():
            param.requires_grad = False
        # Freeze language backbone parameters by default (we selectively unfreeze for fine-tuning)
        for param in self.llm.parameters():
            param.requires_grad = False

        vit_dim = self.vit.config.hidden_size
        llm_dim = self.llm.config.n_embd
        self.projector = VQAProjector(vit_dim=vit_dim, llm_dim=llm_dim, use_mlp=use_mlp)

    def forward(self, pixel_values, input_ids, attention_mask, text_labels=None):
        device = pixel_values.device
        batch_size = pixel_values.size(0)

        # 1. Get visual embeddings from CLIP Vision Encoder
        vit_outputs = self.vit(pixel_values)
        image_features = vit_outputs.last_hidden_state
        num_patches = image_features.size(1)

        # 2. Project visual features to LLM embedding dimension
        visual_embeddings = self.projector(image_features)

        # 3. Get text embeddings from LLM
        text_embeddings = self.llm.transformer.wte(input_ids)

        # 4. Concatenate visual and text embeddings along sequence dimension
        concat_embeds = torch.cat([visual_embeddings, text_embeddings], dim=1)

        # 5. Concatenate attention masks (visual tokens are always visible = 1)
        visual_mask = torch.ones(batch_size, num_patches, dtype=torch.long, device=device)
        concat_mask = torch.cat([visual_mask, attention_mask], dim=1)

        # 6. Setup labels for language modeling loss
        if text_labels is not None:
            visual_labels = torch.full((batch_size, num_patches), -100, dtype=torch.long, device=device)
            concat_labels = torch.cat([visual_labels, text_labels], dim=1)
        else:
            concat_labels = None

        # 7. Setup correct positional embedding IDs sequentially
        concat_position_ids = torch.arange(
            0, num_patches + input_ids.size(1),
            dtype=torch.long,
            device=device
        ).unsqueeze(0).expand(batch_size, -1)

        # 8. Decode using LLM with custom position_ids
        outputs = self.llm(
            inputs_embeds=concat_embeds,
            attention_mask=concat_mask,
            position_ids=concat_position_ids,
            labels=concat_labels,
            return_dict=True
        )

        return outputs

    @torch.no_grad()
    def generate_answer(self, pixel_values, question_input_ids, question_attention_mask, max_new_tokens=20, eos_token_id=50256):
        self.eval()
        device = pixel_values.device
        batch_size = pixel_values.size(0)

        # Extract & project visual features
        vit_outputs = self.vit(pixel_values)
        image_features = vit_outputs.last_hidden_state
        num_patches = image_features.size(1)
        visual_embeddings = self.projector(image_features)

        # Text embeddings
        text_embeddings = self.llm.transformer.wte(question_input_ids)

        current_embeds = torch.cat([visual_embeddings, text_embeddings], dim=1)

        # Combine mask
        visual_mask = torch.ones(batch_size, num_patches, dtype=torch.long, device=device)
        current_mask = torch.cat([visual_mask, question_attention_mask], dim=1)

        # Setup position_ids sequentially
        current_position_ids = torch.arange(
            0, num_patches + question_input_ids.size(1),
            dtype=torch.long,
            device=device
        ).unsqueeze(0).expand(batch_size, -1)

        generated_tokens = []

        for i in range(max_new_tokens):
            outputs = self.llm(
                inputs_embeds=current_embeds,
                attention_mask=current_mask,
                position_ids=current_position_ids,
                return_dict=True
            )

            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated_tokens.append(next_token)

            if next_token.item() == eos_token_id:
                break

            # Update inputs for next iteration
            next_embed = self.llm.transformer.wte(next_token)
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)

            next_mask = torch.ones(batch_size, 1, dtype=torch.long, device=device)
            current_mask = torch.cat([current_mask, next_mask], dim=1)

            next_pos = torch.full((batch_size, 1), num_patches + question_input_ids.size(1) + i, dtype=torch.long, device=device)
            current_position_ids = torch.cat([current_position_ids, next_pos], dim=1)

        return torch.cat(generated_tokens, dim=-1)

    def unfreeze_llm(self):
        for param in self.llm.parameters():
            param.requires_grad = True

    def unfreeze_vit(self):
        for param in self.vit.parameters():
            param.requires_grad = True


## 4. Download and Load VQA v2 Dataset
We download the official `lmms-lab/VQAv2` dataset split via the `datasets` library.
Since the complete dataset is very large (~443k questions for training), we select a slice/subset for quick demonstration training. Adjust `TRAIN_SUBSET_SIZE` to increase the data size as desired!


In [5]:

from datasets import load_dataset

# Stream ONLY the official training parquet files directly from the hub
train_stream = load_dataset(
    "lmms-lab/VQAv2",
    data_files="data/train-*.parquet",
    split="train",
    streaming=True
)

# Stream the official validation parquet files directly
# FIXED: Assigned the output directly to val_stream
val_stream = load_dataset(
    "lmms-lab/VQAv2",
    data_files="data/validation-*.parquet",
    split="train",
    streaming=True
).take(1500)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/962 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/136 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/136 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

## 5. Tokenizer, ImageProcessor, and DataLoaders Setup
Initialize processing layers and set up PyTorch `DataLoader` objects.


In [6]:

import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer, CLIPImageProcessor

vit_model_name = 'openai/clip-vit-base-patch32'
llm_model_name = 'gpt2'

print('Loading pre-trained tokenizers and processors...')
tokenizer = GPT2Tokenizer.from_pretrained(llm_model_name)
tokenizer.pad_token = tokenizer.eos_token
image_processor = CLIPImageProcessor.from_pretrained(vit_model_name)

print('Converting validation streaming dataset to in-memory list...')
# FIXED: Changed from val_ds to val_stream
val_ds_mapped = list(val_stream)

# Initialize validation dataset
val_dataset = VQAv2Dataset(val_ds_mapped, image_processor, tokenizer)

# Dataloader
BATCH_SIZE = 16
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('Validation dataloader ready!')


Loading pre-trained tokenizers and processors...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Converting validation streaming dataset to in-memory list...
Validation dataloader ready!


## 6. Validation Evaluation Function
We compute the exact match accuracy of the generated responses against the ground-truth answers.


In [7]:

import re

def normalize_answer(s):
    s = s.lower().strip()
    # Remove articles
    s = re.sub(r'\b(a|an|the)\b', '', s)
    # Remove punctuation
    s = re.sub(r'[^\w\s]', '', s)
    # Clean whitespace
    s = " ".join(s.split())
    return s

def evaluate(model, dataloader, tokenizer, device):
    model.eval()
    correct = 0
    total = 0

    print('Running evaluation...')
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            pixel_values = batch['pixel_values'].to(device)
            questions = batch['question']
            answers = batch['answer']

            # UPGRADE: Added a trailing space after 'Answer:' to match prompt format in VQAv2Dataset
            prompts = [f'Question: {q} Answer: ' for q in questions]
            inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
            input_ids = inputs['input_ids'].to(device)
            attention_mask = inputs['attention_mask'].to(device)

            for i in range(pixel_values.size(0)):
                pred_tokens = model.generate_answer(
                    pixel_values=pixel_values[i:i+1],
                    question_input_ids=input_ids[i:i+1],
                    question_attention_mask=attention_mask[i:i+1],
                    max_new_tokens=10,
                    eos_token_id=tokenizer.eos_token_id
                )

                pred_text = normalize_answer(tokenizer.decode(pred_tokens[0], skip_special_tokens=True))
                gt_text = normalize_answer(answers[i])

                if pred_text == gt_text:
                    correct += 1
                total += 1

    accuracy = (correct / total) * 100 if total > 0 else 0
    return accuracy


## 7. Model Instantiation & Training Loop
Initialize the model on the GPU and execute the training loop. We save the best model weights to `vqa_fusion_model_v2.pt` when validation accuracy improves.


In [ ]:

import os
import gc
# UPGRADE: Fine-tune the LLM for high accuracy with optimized learning rates
EPOCHS = 3
LEARNING_RATE = 1e-4       # Learning rate for projection head
LLM_LEARNING_RATE = 2e-5   # Stable lower learning rate for GPT-2 layers
UNFREEZE_LLM = True        # Set to True to unfreeze and fine-tune GPT-2 layers
SAVE_PATH = 'checkpoints/vqa_fusion_model.pt'

CHUNK_SIZE = 10000         # Train on chunks of 10k samples sequentially

print('Instantiating model on device...')
model = ViTLLMFusionModel(
    vit_model_name=vit_model_name,
    llm_model_name=llm_model_name,
    use_mlp=True
)

if UNFREEZE_LLM:
    print('Unfreezing LLM parameters...')
    model.unfreeze_llm()
    optimizer_params = [
        {'params': model.projector.parameters(), 'lr': LEARNING_RATE},
        {'params': model.llm.parameters(), 'lr': LLM_LEARNING_RATE}
    ]
else:
    print('Training projection head only...')
    optimizer_params = model.projector.parameters()

model.to(device)
optimizer = torch.optim.AdamW(optimizer_params, lr=LEARNING_RATE)

best_val_acc = -1.0
VAL_INTERVAL_STEPS = 2000          # Run validation evaluation & save check every 2000 steps
GRADIENT_ACCUMULATION_STEPS = 1    # Increase to 2, 4, etc. to optimize VRAM (effective batch size = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

# Setup GradScaler for Automatic Mixed Precision (AMP) to save VRAM and speed up training
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

for epoch in range(EPOCHS):
    print(f'\n================ Starting Epoch {epoch+1}/{EPOCHS} ================')

    # Shuffle training stream differently each epoch using a light memory buffer
    shuffled_train_stream = train_stream.shuffle(buffer_size=1000, seed=42 + epoch)

    # Initialize the streaming dataset and dataloader
    train_dataset = VQAStreamingDataset(shuffled_train_stream, image_processor, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=2)

    model.train()
    epoch_loss = 0.0
    steps = 0

    optimizer.zero_grad()
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1} Training')

    for batch in progress_bar:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass with Automatic Mixed Precision (AMP) for VRAM efficiency
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                text_labels=labels
            )
            loss = outputs.loss
            # Scale loss for gradient accumulation
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        # Backward pass with scaled gradients
        scaler.scale(loss).backward()

        # Optimizer step (respecting gradient accumulation steps)
        if (steps + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        steps += 1
        progress_bar.set_postfix({'loss': f'{(loss.item() * GRADIENT_ACCUMULATION_STEPS):.4f}'})

        # Periodic validation & saving best checkpoint to avoid losing progress
        if steps % VAL_INTERVAL_STEPS == 0:
            print(f'\nStep {steps}: Running periodic evaluation on validation set...')
            val_acc = evaluate(model, val_loader, tokenizer, device)
            print(f'Validation Accuracy at Step {steps}: {val_acc:.2f}%')

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                print(f'Saving new best checkpoint to {SAVE_PATH}')
                os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
                torch.save(model.state_dict(), SAVE_PATH)

            model.train()

        # Regular memory cleanup
        if steps % 100 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Step at the end of epoch if any accumulated gradients are remaining
    if steps % GRADIENT_ACCUMULATION_STEPS != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_loss = epoch_loss / steps if steps > 0 else 0
    print(f'Epoch {epoch+1} complete. Average loss: {avg_loss:.4f}')

    # Evaluate validation accuracy at the end of epoch
    val_acc = evaluate(model, val_loader, tokenizer, device)
    print(f'Epoch {epoch+1} Final Validation Accuracy: {val_acc:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f'Saving new best checkpoint to {SAVE_PATH}')
        os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
        torch.save(model.state_dict(), SAVE_PATH)

print(f'Training complete! Best accuracy: {best_val_acc:.2f}%')


Instantiating model on device...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1

...

Evaluating:   0%|          | 0/94 [00:00<?, ?it/s]

Validation Accuracy at Step 10000: 21.53%

Step 12000: Running periodic evaluation on validation set...
Running evaluation...


Evaluating:   0%|          | 0/94 [00:00<?, ?it/s]

Validation Accuracy at Step 12000: 26.07%
Saving new best checkpoint to checkpoints/vqa_fusion_model.pt

Step 14000: Running periodic evaluation on validation set...
Running evaluation...


Evaluating:   0%|          | 0/94 [00:00<?, ?it/s]

## 8. Interactive Inference Check
Pick a sample from the validation dataset, render it, and generate answers dynamically to verify the trained fusion model's behavior!


In [ ]:

import matplotlib.pyplot as plt
import os
import torch
import random

# Load the best weights if they exist
if os.path.exists(SAVE_PATH):
    print('Loading fine-tuned model checkpoint weights...')
    model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

# Define how many different samples you want to check
NUM_SAMPLES_TO_CHECK = 3

# Pick completely random indices from the validation set
sample_indices = random.sample(range(len(val_ds_mapped)), NUM_SAMPLES_TO_CHECK)

for idx, sample_idx in enumerate(sample_indices):
    print(f"\n================ Checking Sample {idx + 1} (Random Index: {sample_idx}) ================")

    # Extract data from the mapped list
    sample_item = val_ds_mapped[sample_idx]
    test_image = sample_item['image']
    test_question = sample_item['question']
    true_answer = sample_item['multiple_choice_answer']

    # Show the image with question
    plt.figure(figsize=(5, 5))
    plt.imshow(test_image)
    plt.title(f'Sample {idx + 1} Question: {test_question}')
    plt.axis('off')
    plt.show()

    # Prepare tokens
    pixel_values = image_processor(images=test_image, return_tensors='pt').pixel_values.to(device)
    # UPGRADE: Kept original casing and trailing space to match VQAv2Dataset prompts perfectly
    prompt_text = f'Question: {test_question} Answer: '
    prompt_enc = tokenizer(prompt_text, return_tensors='pt', add_special_tokens=False)
    question_input_ids = prompt_enc['input_ids'].to(device)
    question_attention_mask = prompt_enc['attention_mask'].to(device)

    # Generate model prediction
    generated_ids = model.generate_answer(
        pixel_values=pixel_values,
        question_input_ids=question_input_ids,
        question_attention_mask=question_attention_mask,
        max_new_tokens=10,
        eos_token_id=tokenizer.eos_token_id
    )

    predicted_answer = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

    print('-----------------------------------------')
    print(f'Question: {test_question}')
    print(f'Predicted Answer:  \033[1m{predicted_answer}\033[0m')
    print(f'Ground-Truth Answer: {true_answer}')
    print('-----------------------------------------\n')

In [ ]:
!zip -r checkpoints.zip /kaggle/working/checkpoints